# Multi-horizon TCN — A-share trend prediction

Trains a Temporal Convolutional Network that outputs, **per horizon (1/5/20 days)**, a softmax over directions `{down, flat, up}`.

**How to run via the VS Code Google Colab extension:**
1. Open this notebook in VS Code.
2. Top-right kernel picker → connect to a **Colab** runtime (pick a **GPU** runtime for speed).
3. Run the cells top to bottom.

The Colab runtime is a fresh cloud VM, so it does **not** see your local repo. Set `REPO_URL` below to your pushed GitHub repo (or upload the `src/` folder). If you run this notebook on a *local* kernel instead, it imports `src` directly.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = ""  # e.g. "https://github.com/<you>/Peng.git" — leave blank if src/ is already present

if IN_COLAB and REPO_URL and not os.path.exists("Peng"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("Peng")
elif not os.path.exists("src") and os.path.exists("../src"):
    os.chdir("..")  # running from notebooks/ on a local kernel

print("cwd:", os.getcwd(), "| in_colab:", IN_COLAB)
assert os.path.exists("src"), "src/ not found — set REPO_URL or upload the package"

In [ ]:
!pip -q install -r requirements.txt
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
from src.config import Config
from src.train import train

cfg = Config(
    symbol="600519",        # Kweichow Moutai; change to your A-share code
    horizons=[1, 5, 20],
    window=60,
    epochs=60,
)
ckpt_path = train(cfg)

## Inference — per-horizon direction probabilities

Load the best checkpoint and print softmax probabilities for the most recent window.

In [ ]:
import numpy as np, torch
from src.data import load_prices
from src.features import build_dataset_frame
from src.model import MultiHorizonTCN

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
saved = ckpt["config"]

model = MultiHorizonTCN(
    num_features=len(ckpt["scaler_mean"]),
    horizons=saved["horizons"],
    num_classes=saved["num_classes"],
    channels=saved["channels"],
    kernel_size=saved["kernel_size"],
    dropout=saved["dropout"],
)
model.load_state_dict(ckpt["model_state"])

prices = load_prices(saved["symbol"], saved["start_date"], saved["end_date"], saved["adjust"], saved["cache_dir"])
feats, _ = build_dataset_frame(prices, saved["horizons"], saved["flat_threshold"])

# Scale the last `window` rows with the training scaler, shape (1, F, window).
window = saved["window"]
x = (feats.to_numpy()[-window:] - ckpt["scaler_mean"]) / ckpt["scaler_scale"]
x = torch.tensor(x.T[None], dtype=torch.float32)

probs = model.predict_proba(x)
classes = ["down", "flat", "up"]
print(f"As of {feats.index[-1].date()} ({saved['symbol']}):\n")
for h, p in zip(saved["horizons"], probs):
    p = p[0].numpy()
    pretty = "  ".join(f"{c}={v:.2f}" for c, v in zip(classes, p))
    print(f"  {h:>2}d  ->  {pretty}   (argmax: {classes[int(np.argmax(p))]})")